# This notebook serves to prepare/explore the dataset. Plus, creating a cleaned, lightly filtered dataset in a form of a json file. 

### Loading the data

In [1]:
import gzip
import json

with gzip.open('results.json.gz', 'rt', encoding='utf-8') as f:
    articles = json.load(f)

num = 1

print(f"Total articles: {len(articles)}")
print(f"\nSample article keys: {list(articles[num].keys())}")
print(f"\nFirst article preview:")
print(f"  Title:  {articles[num].get('title', 'N/A')}")
print(f"  Date:   {articles[num].get('date', 'N/A')}")
print(f"  Source: {articles[num].get('source', 'N/A')}")
print(f"  Url:    {articles[num].get('url', 'N/A')}")

import textwrap

text_preview = articles[num].get('text', '')
wrapped = textwrap.fill(text_preview, width=80)
print(f"  Text:\n\n{wrapped}")

Total articles: 8610

Sample article keys: ['title', 'date', 'text', 'source', 'goid', 'url']

First article preview:
  Title:  ADMINISTRATIVE PITFALLS OF CYBER INSURANCE POLICIES
  Date:   2023-04-01
  Source: digital
  Url:    https://www.proquest.com/docview/2791686136
  Text:

  How a cyber policy can make insurance brokers more vulnerable to costly
mistakes  The first cyber insurance policy was written back in 1997. By popular
account, it was developed for AIG and called an "internet security liability
policy." It was geared toward information technology companies in the business
of managing the networks and systems of other businesses and consumers. Since
then, the internet and cyber capabilities have gotten far more sophisticated and
complex, becoming a global communications force for how we work and play. At the
same time, its vast expansion has opened the doors to significant risk when
valuable data are stored and transferred digitally. Cyber criminals are
increasingly taking 

# Cleaning out the dataset 

In [2]:
import re 

def clean_text(text):
    text = re.sub(r'http\S+|www\.\S+', '', text)           # remove URLs
    text = re.sub(r'([.!?])"([A-Z])', r'\1" \2', text)    # fix ."A quote boundaries
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)      # fix stuck words
    text = re.sub(r'-\s*\n\s*', '', text)                  # fix hyphenated line breaks
    text = re.sub(r'\.{2,}', '.', text)                    # fix repeated periods
    text = re.sub(r',{2,}', ',', text)                     # fix repeated commas
    text = re.sub(r'\s+', ' ', text).strip()               # collapse whitespace
    return text


# apply to all articles
for a in articles:
    a['text'] = clean_text(a.get('text', ''))

print("Cleaning done.")
print(textwrap.fill(articles[0]['text'], width=80))


Cleaning done.
Changes expected as employers, employees deal with pandemic-related and other
concerns Teetering on the edge of elimination for the past four years, the
Affordable Care Act is likely to be not only rescued but expanded in a Joe Biden
administration; and with greater attention paid to health issues post COVID-19
pandemic, employee health benefits may be entering their most complicated era of
evolution. President-elect Biden promised voters a revised national health
insurance program, including a public option that could be an alternative to
employer-paid insurance. But in the meantime, as legislators grapple with that
promise, benefits insurers predict that 2021 will feature plenty of other
changes. More options More flexible supplemental voluntary benefits, new
benefits communication technology, and new open enrollment processes designed to
reach employees who continue to work away from crowded offices already are in
development. Employee benefits underwriters also are p

In [3]:
focus_keywords = [
    "commission", "client", "trust", "recommend", "policyholder",
    "retention", "disclosure", "fiduciary", "hard market", "soft market",
    "market withdrawal", "reinsurance", "surplus lines", "catastrophe",
    "hurricane", "wildfire", "flood", "rate increase", "non-renewal",
    "uninsurable", "FAIR Plan", "residual market", "transparency"
]

import re

kw_patterns = {kw: re.compile(r'\b' + re.escape(kw) + r'\b', re.IGNORECASE) for kw in focus_keywords}

def get_matched_keywords(text):
    matched = []
    for kw, pattern in kw_patterns.items():
        if pattern.search(text):
            matched.append(kw)
    return matched

filtered_articles = []
for a in articles:
    text = a.get('text', '')
    matched = get_matched_keywords(text)
    if matched:
        filtered_articles.append({
            'title': a.get('title', ''),
            'url': a.get('url', ''),
            'date': a.get('date', ''),
            'text': text,
            'source': "rough notes",
            'matched_keywords': matched
        })

print(f"Total articles: {len(articles)}")
print(f"Articles containing at least one focus keyword: {len(filtered_articles)}")
print(f"Articles filtered out: {len(articles) - len(filtered_articles)}")

with open('cleaned_articles.json', 'w', encoding='utf-8') as f:
    json.dump(filtered_articles, f, indent=2)

print(f"\nSaved to cleaned_articles.json")

Total articles: 8610
Articles containing at least one focus keyword: 6275
Articles filtered out: 2335

Saved to cleaned_articles.json


## manual review of 15 articles 

In [4]:
import random

random.seed(30)
sample = random.sample(filtered_articles, 15)

with open('manual_review_15.json', 'w', encoding='utf-8') as f:
    json.dump(sample, f, indent=2)

print(f"Saved to manual_review_15.json")
print(f"Total sampled: {len(sample)}")

Saved to manual_review_15.json
Total sampled: 15
